# Inference

Scores eligible clients at an inference date, **promotes** the models it uses into a curated store, and writes **batch-scoped** scores so every scoring run is a self-contained, reproducible unit.

- **Local scratch runs** (`MODELS_DIR`, cwd-relative `artifacts/`) — many training runs; you pick.
- **Promoted model store** (`06_models`) — the used model is copied to `<store>/<product>/<run_id>/`; the audit / rerun / reinfer source of truth.
- **Model output** (`07_model_output/<scoring_run_id>/`) — this batch's `scores/` **and** its `manifest.json`, side by side. Each batch is immutable; nothing overwrites a prior batch.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))

from src import _compat  # restores np.NaN for pandas-on-Spark (NumPy 2); MUST precede pyspark/src
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from src.inference import (discover_products, resolve_run, load_scorer, score_product,
                           promote_model, write_manifest, verify_model,
                           resolve_from_manifest, _read_text)

### Parameters
- `MODELS_DIR`: local scratch runs to score from (read).
- `MODELS_STORE` / `OUTPUT_ROOT`: your `06_models` / `07_model_output` (set `REPO_PARENT` for HDFS).
- `PROMOTE`: promote the used model into `MODELS_STORE` (default `True`).
- `RUN_ID`: `None` (latest per product) · a `str` (that run for all) · a `dict` `{product: run_id}` (pin some, latest for the rest).
- `PRODUCTS` / `INFER_DATE`: `None` = all local products / each product's frozen date.

In [ ]:
import pandas as pd, pathlib, datetime, json

REPO_PARENT  = '.'                                      # set to {hdfs_repo_parent} for HDFS
MODELS_DIR   = 'artifacts'                              # local scratch runs (READ)
MODELS_STORE = f'{REPO_PARENT}/06_models'               # promoted store (WRITE)
OUTPUT_ROOT  = f'{REPO_PARENT}/07_model_output'         # scores + manifest (WRITE)
PROMOTE      = True
INFER_DATE   = None        # None -> per-product frozen date; or e.g. '2026-08-01'
RUN_ID       = None        # None | 'run_id' | {'home_loans': 'run_id', ...}
PRODUCTS     = None        # None -> all local products; or ['fx_activation', ...]

scoring_run_id = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
SCORES_PATH    = f'{OUTPUT_ROOT}/{scoring_run_id}/scores'   # batch-scoped (Option 1)
products = PRODUCTS or discover_products(MODELS_DIR)
assert products, f'no local runs under {MODELS_DIR}'
print('batch', scoring_run_id, '|', len(products), 'products | promote ->',
      MODELS_STORE if PROMOTE else 'OFF')

### Score → promote → manifest
Each product is scored and promoted inside the loop; the scores are **collected and written once** after the loop (`unionByName` → single `overwrite` of this batch's `07_model_output/<scoring_run_id>/scores`). Prior batches are untouched, and a failure at the write leaves no half-written batch. `RUN_ID` resolves per product (dict) or uniformly (None/str).

In [ ]:
from functools import reduce

summary, frames = [], []
for i, product in enumerate(products, 1):
    try:
        rid = RUN_ID.get(product) if isinstance(RUN_ID, dict) else RUN_ID
        run_dir = resolve_run(product, MODELS_DIR, run_id=rid)
        cfg, model, calibrate, model_hash = load_scorer(run_dir)
        idate = INFER_DATE or str(cfg.infer_date)
        long = score_product(spark, cfg, model, calibrate, idate, model_hash)
        frames.append(spark.createDataFrame(long))   # cols: cif, product, model_hash, infer_date, score
        promoted = promote_model(run_dir, MODELS_STORE, product, run_dir.name) if PROMOTE else None
        summary.append({'product': product, 'status': 'ok', 'run_id': run_dir.name,
                        'model_hash': model_hash, 'infer_date': idate, 'rows': len(long),
                        'promoted_dir': promoted})
        print(f'[{i}/{len(products)}] {product}: {len(long):,} rows | run {run_dir.name}')
    except Exception as e:
        summary.append({'product': product, 'status': f'SKIPPED: {e}', 'rows': 0})
        print(f'[{i}/{len(products)}] {product}: SKIPPED - {e}')

# single write of the whole batch (all products) — no per-product overwrite
if frames:
    (reduce(lambda a, b: a.unionByName(b), frames)
        .write.mode('overwrite').partitionBy('product').parquet(SCORES_PATH))

manifest = write_manifest(f'{OUTPUT_ROOT}/{scoring_run_id}', {
    'scoring_run_id': scoring_run_id, 'scores_path': SCORES_PATH,
    'models_dir': MODELS_STORE, 'products': summary})
print('manifest:', manifest)
pd.DataFrame(summary)

### Verify this batch's scores
Also asserts that every product marked `ok` in the summary is actually present on disk.

In [ ]:
out = spark.read.parquet(SCORES_PATH)
print('rows in this batch:', out.count())
out.groupBy('product').count().orderBy('product').show()
out.select('cif', 'product', 'model_hash', 'infer_date', 'score').show(5)

# every product scored 'ok' must actually be on disk (catches a partial / overwritten batch)
ok_products = {s['product'] for s in summary if s['status'] == 'ok'}
on_disk = {r['product'] for r in out.select('product').distinct().collect()}
assert on_disk == ok_products, f'scored ok but missing on disk: {ok_products - on_disk}; unexpected: {on_disk - ok_products}'
print('all', len(ok_products), 'scored products present on disk')

## Loading from a manifest
`manifest` is the path `write_manifest` returned; in a fresh session set it to a past batch, e.g. `manifest = f'{OUTPUT_ROOT}/<scoring_run_id>/manifest.json'`. `_read_text` is HDFS-aware; for a local/mounted path `pathlib.Path(manifest).read_text()` also works.

### The full manifest

In [ ]:
man = json.loads(_read_text(manifest))
print(json.dumps(man, indent=2))

### One product's entry (which promoted model produced its scores)

In [ ]:
prod = man['products'][0]['product']
next(r for r in man['products'] if r['product'] == prod)

### All scores belonging to this manifest
Because scores are batch-scoped, the manifest's `scores_path` *is* this batch's scores — no cross-batch filtering needed.

In [ ]:
batch_scores = spark.read.parquet(man['scores_path'])
print('rows:', batch_scores.count())
batch_scores.groupBy('product', 'model_hash').count().orderBy('product').show()
batch_scores.show(5)

### Reproduce a promoted model (audit)
Resolves the `run_id` in `06_models`, fetches it local if on HDFS, and **verifies it still hashes to what produced the scores** before loading.

In [ ]:
run_dir, cfg2, model2, calibrate2, mh = resolve_from_manifest(
    manifest, prod, models_dir=man['models_dir'])
print('reproduced', prod, '| promoted run', run_dir.rstrip('/').split('/')[-1],
      '| hash verified:', mh)

## Reinfer — rescore promoted models on a new date / input table
The point of the promoted store + manifest: you can rescore **without the local scratch runs**. Load the models the manifest points at, then score a fresh month (and optionally a different source table with the *same schema*). Set `PERSIST_AS_BATCH=True` to write it as a new, self-contained batch that references the same promoted models.

In [ ]:
NEW_INFER_DATE   = '2026-09-01'   # a later month to score
NEW_TABLE        = None           # None -> same table; or a different table with the SAME schema
PERSIST_AS_BATCH = False          # True -> write a new scored batch + manifest

from functools import reduce
man = json.loads(_read_text(manifest))
new_run_id  = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
new_scores  = f'{OUTPUT_ROOT}/{new_run_id}/scores'
rows, frames = [], []
for r in [p for p in man['products'] if p.get('status') == 'ok']:
    product = r['product']
    rd, cfg_r, model_r, cal_r, mh_r = resolve_from_manifest(manifest, product,
                                                            models_dir=man['models_dir'])
    if NEW_TABLE:
        cfg_r = cfg_r.model_copy(update={'table': NEW_TABLE})   # same schema, new source
    long = score_product(spark, cfg_r, model_r, cal_r, NEW_INFER_DATE, mh_r)
    print(product, '->', f'{len(long):,}', 'rows rescored @', NEW_INFER_DATE)
    if PERSIST_AS_BATCH:
        frames.append(spark.createDataFrame(long))
        rows.append({'product': product, 'status': 'ok', 'run_id': rd.rstrip('/').split('/')[-1],
                     'model_hash': mh_r, 'infer_date': NEW_INFER_DATE, 'rows': len(long),
                     'promoted_dir': rd})

if PERSIST_AS_BATCH and frames:
    (reduce(lambda a, b: a.unionByName(b), frames)
        .write.mode('overwrite').partitionBy('product').parquet(new_scores))
    m2 = write_manifest(f'{OUTPUT_ROOT}/{new_run_id}', {
        'scoring_run_id': new_run_id, 'scores_path': new_scores, 'models_dir': man['models_dir'],
        'reinfer_of': man['scoring_run_id'], 'products': rows})
    print('new batch manifest:', m2)

### Notes
- **Batch-scoped scores:** each run is `07_model_output/<scoring_run_id>/{scores, manifest.json}` — immutable, self-contained, and the manifest sits next to the scores it describes.
- **Reinfer needs only `06_models` + the manifest** — not the local scratch runs. That is what makes the promoted store worth keeping.
- **HDFS:** promotion/manifest writes dispatch to `hdfs dfs`; reproduction fetches the promoted bundle local first. A mounted store avoids all of this. The `hdfs dfs` branches are untested offline — verify in your environment.